----------------------------------------------------------------------------------------------
--------------------------------- ROW REMOVAL -------------------------
----------------------------------------------------------------------------------------------


In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
import copy
import os

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


/Users/elif/Desktop/Projects/privacy_in_ml/jupyter_notebooks


In [5]:
bean_train = pd.read_csv("../datasets/bean_train.csv")
bean_test = pd.read_csv("../datasets/bean_test.csv")

X_train = bean_train.drop('Class', axis=1)
X_test = bean_test.drop('Class', axis=1)
y_train = bean_train['Class']
y_test = bean_test['Class']


In [6]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

output_file = "../results/row_removal_bean.xlsx"
n_iter = 10
epsilon_values = [0.1, 1, 5, 10, 30, 50, 100]

### Baseline (full data, no removal, no perturbation)

In [7]:
baseline_model = LogisticRegression(
    penalty="l2",
    C=1,
    max_iter=1000,
    random_state=42
)
baseline_model.fit(X_train_sc, y_train)

fc.predict_multiclass(baseline_model, X_train_sc, y_train, conf_matrix=False)

fc.predict_multiclass_save_results(
    baseline_model,
    X_test_sc,
    y_test,
    conf_matrix=False,
    perturbation_type="Original",
    epsilon=0,
    row_id=None,
    output_file=output_file
)


---------------------------------------
Accuracy:  0.9245
Precision: 0.9249
Recall:    0.9245
F1 Score:  0.9246
---------------------------------------
---------------------------------------
Accuracy:  0.9266
Precision: 0.9278
Recall:    0.9266
F1 Score:  0.9269
---------------------------------------
Saved results to ../results/row_removal_bean.xlsx


----------------------------------------------------------------------------------------------
-------------------------------- INPUT PERTURBATION (row removal) ----------------------------
----------------------------------------------------------------------------------------------

In [8]:
ranges = [[10000, 300000], [500, 2000], [100, 750], [100, 750],
       [1,3], [0,1], [10000, 300000], [100, 700], [0,1],
       [0,1], [0.2,1], [0.5,1], [0, 0.2], [0, 0.1],
       [0.2,1], [0.9, 1]]
input_sensitivity = []
for r in ranges:
    input_sensitivity.append(r[1] - r[0])


In [9]:
row_rng = np.random.default_rng(42) 
np.random.seed(42)       

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    for e in epsilon_values:
        X_train_perturbed = X_train_reduced.copy()

        for i, var in enumerate(X_train_reduced.columns):
            eps_j = e  # The paper uses eps_j = e/d, but this leads to huge errors
            scale = input_sensitivity[i] / eps_j
            noise = np.random.laplace(loc=0, scale=scale, size=len(X_train_reduced))

            lower, upper = ranges[i]
            X_train_perturbed[var] = (
                X_train_reduced[var] + noise
            ).clip(lower=lower, upper=upper)

        scaler = StandardScaler()
        X_train_perturbed_sc = scaler.fit_transform(X_train_perturbed)
        X_test_perturbed_sc = scaler.transform(X_test)

        input_model = copy.deepcopy(baseline_model)
        input_model.fit(X_train_perturbed_sc, y_train_reduced)

        print(f"Input - iteration {iteration}/{n_iter}, epsilon={e} - removed row {removed_label}")
        fc.predict_multiclass(input_model, X_train_perturbed_sc, y_train_reduced, conf_matrix=False)

        fc.predict_multiclass_save_results(
            input_model,
            X_test_perturbed_sc,
            y_test,
            conf_matrix=False,
            perturbation_type="Input",
            epsilon=e,
            row_id=removed_label,
            output_file=output_file
        )


Input - iteration 1/10, epsilon=0.1 - removed row 971
---------------------------------------
Accuracy:  0.2643
Precision: 0.6627
Recall:    0.2643
F1 Score:  0.1109
---------------------------------------
---------------------------------------
Accuracy:  0.2464
Precision: 0.8143
Recall:    0.2464
F1 Score:  0.0974
---------------------------------------
Saved results to ../results/row_removal_bean.xlsx
Input - iteration 1/10, epsilon=1 - removed row 971
---------------------------------------
Accuracy:  0.2734
Precision: 0.3235
Recall:    0.2734
F1 Score:  0.1830
---------------------------------------
---------------------------------------
Accuracy:  0.2464
Precision: 0.5030
Recall:    0.2464
F1 Score:  0.1000
---------------------------------------
Saved results to ../results/row_removal_bean.xlsx
Input - iteration 1/10, epsilon=5 - removed row 971
---------------------------------------
Accuracy:  0.4573
Precision: 0.4372
Recall:    0.4573
F1 Score:  0.4374
----------------------

----------------------------------------------------------------------------------------------
-------------------------------- OUTPUT PERTURBATION (row removal) ---------------------------
----------------------------------------------------------------------------------------------

In [10]:
row_rng = np.random.default_rng(42) #so that we have the same 10 rows as above
np.random.seed(42)

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    scaler = StandardScaler()
    X_train_reduced_sc = scaler.fit_transform(X_train_reduced)
    X_test_reduced_sc = scaler.transform(X_test)

    reduced_model = copy.deepcopy(baseline_model)
    reduced_model.fit(X_train_reduced_sc, y_train_reduced)

    n = X_train_reduced.shape[0]
    lambda_reg = 1 / reduced_model.C
    output_sensitivity = 2 / (lambda_reg * n) 

    for e in epsilon_values:
        coef_noise = np.random.laplace(loc=0, scale=output_sensitivity / e, size=reduced_model.coef_.shape)
        intercept_noise = np.random.laplace(loc=0, scale=output_sensitivity / 3, size=reduced_model.intercept_.shape)

        output_model = copy.deepcopy(reduced_model)
        output_model.coef_ = reduced_model.coef_ + coef_noise
        output_model.intercept_ = reduced_model.intercept_ + intercept_noise

        print(f"Output - iteration {iteration}/{n_iter}, epsilon={e} - removed row {removed_label}")
        fc.predict_multiclass(output_model, X_train_reduced_sc, y_train_reduced, conf_matrix=False)

        fc.predict_multiclass_save_results(
            output_model,
            X_test_reduced_sc,
            y_test,
            conf_matrix=False,
            perturbation_type="Output",
            epsilon=e,
            row_id=removed_label,
            output_file=output_file
        )


Output - iteration 1/10, epsilon=0.1 - removed row 971
---------------------------------------
Accuracy:  0.9246
Precision: 0.9250
Recall:    0.9246
F1 Score:  0.9247
---------------------------------------
---------------------------------------
Accuracy:  0.9262
Precision: 0.9275
Recall:    0.9262
F1 Score:  0.9266
---------------------------------------
Saved results to ../results/row_removal_bean.xlsx
Output - iteration 1/10, epsilon=1 - removed row 971
---------------------------------------
Accuracy:  0.9246
Precision: 0.9249
Recall:    0.9246
F1 Score:  0.9247
---------------------------------------
---------------------------------------
Accuracy:  0.9266
Precision: 0.9278
Recall:    0.9266
F1 Score:  0.9269
---------------------------------------
Saved results to ../results/row_removal_bean.xlsx
Output - iteration 1/10, epsilon=5 - removed row 971
---------------------------------------
Accuracy:  0.9245
Precision: 0.9249
Recall:    0.9245
F1 Score:  0.9246
-------------------

----------------------------------------------------------------------------------------------
-------------------------------- INTERNAL PERTURBATION (row removal) -------------------------
----------------------------------------------------------------------------------------------

In [11]:
row_rng = np.random.default_rng(42)   # reset: same 10 rows as Input/Output above

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    scaler = StandardScaler()
    X_train_reduced_sc = scaler.fit_transform(X_train_reduced)
    X_test_reduced_sc = scaler.transform(X_test)

    data_norm = np.max(np.linalg.norm(X_train_reduced_sc, axis=1))

    print(f"Internal - iteration {iteration}/{n_iter} - removed row {removed_label}, data_norm={data_norm:.4f}")

    fc.internal_perturbation_save_results(
        "bean_row_removal",
        X_train_reduced_sc,
        y_train_reduced,
        X_test_reduced_sc,
        y_test,
        epsilon_values=epsilon_values,
        data_norm=data_norm,
        C=baseline_model.C,
        bivariate=False,
        row_id=removed_label,
        output_file=output_file,
        perturbation_type="Internal"
    )


Internal - iteration 1/10 - removed row 971, data_norm=18.6881
---------------------------------------
Epsilon: 0.1
Accuracy: 0.0896
Precision: 0.0728
Recall: 0.0896
F1 Score: 0.0744
Time: 0.03 s
---------------------------------------
---------------------------------------
Epsilon: 1
Accuracy: 0.3371
Precision: 0.2567
Recall: 0.3371
F1 Score: 0.2840
Time: 0.04 s
---------------------------------------
---------------------------------------
Epsilon: 5
Accuracy: 0.6974
Precision: 0.7012
Recall: 0.6974
F1 Score: 0.6946
Time: 0.06 s
---------------------------------------
---------------------------------------
Epsilon: 10
Accuracy: 0.8057
Precision: 0.8161
Recall: 0.8057
F1 Score: 0.8083
Time: 0.06 s
---------------------------------------
---------------------------------------
Epsilon: 30
Accuracy: 0.7965
Precision: 0.8433
Recall: 0.7965
F1 Score: 0.8049
Time: 0.07 s
---------------------------------------
---------------------------------------
Epsilon: 50
Accuracy: 0.9045
Precision